[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hentablihamza/practical-biopython-course/blob/main/notebooks/01_python_refresher_biopython_setup.ipynb)

# 01 · Python Refresher and Biopython Setup

*Practical Biopython Course · Hamza Hentabli, Ph.D. · ~40 min*

Bioinformatics code is mostly ordinary Python applied to sequences and tables. Before using
Biopython, this notebook refreshes the handful of Python skills you will need in every later lesson,
using real data: a panel of **15 protein kinases** from UniProt, including the cancer drug target
**EGFR** (epidermal growth factor receptor), which we follow throughout the course.

**Learning objectives.** By the end of this notebook you will be able to:

- use strings, lists, dictionaries, loops, and functions on biological sequences
- read a small data table with pandas and summarize it
- install and import Biopython, and check its version
- read your first sequence file with `Bio.SeqIO`

In [ ]:
# Setup: installs dependencies on Google Colab and locates the course data.
# Running locally? Install requirements.txt once and this cell does nothing extra.
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", 'biopython'], check=True)
    if not os.path.isdir("practical-biopython-course"):
        subprocess.run(["git", "clone", "-q", "--depth", "1",
                        "https://github.com/hentablihamza/practical-biopython-course.git"], check=True)
    DATA_DIR = "practical-biopython-course/data"
else:
    DATA_DIR = "../data" if os.path.isdir("../data") else "data"

print("Running on Colab:", IN_COLAB, "| data folder:", DATA_DIR)

## 1. A sequence is just a string

DNA, RNA, and protein sequences can be stored as plain Python strings. Indexing starts at **0**,
while biologists number residues from **1**. Keep this off-by-one difference in mind; it causes
more bugs in bioinformatics than anything else.

In [ ]:
dna = "ATGCGACCCTCCGGGACGGCCGGGGCAGCGCTCCTGGCG"   # first 39 nt of the human EGFR coding sequence
print("length:", len(dna))
print("first codon:", dna[0:3])
print("last 6 nt:", dna[-6:])
print("is 'GGG' present?", "GGG" in dna, "| how many G:", dna.count("G"))

## 2. Loops, dictionaries, and functions

A dictionary maps keys to values; it is the natural structure for counting.

In [ ]:
def base_counts(seq):
    """Count each nucleotide in a DNA string."""
    counts = {}
    for base in seq:
        counts[base] = counts.get(base, 0) + 1
    return counts

def gc_percent(seq):
    """Percentage of G and C bases."""
    return 100 * (seq.count("G") + seq.count("C")) / len(seq)

print(base_counts(dna))
print(f"GC content: {gc_percent(dna):.1f}%")

In [ ]:
# Split a coding sequence into codons with a list comprehension
codons = [dna[i:i + 3] for i in range(0, len(dna) - len(dna) % 3, 3)]
print(codons)

## 3. Tables with pandas

Metadata about sequences (organism, length, domain boundaries) lives in tables. The file
`data/sequences/kinase_panel.csv` was downloaded from the UniProt REST API (see `data/README.md`).

In [ ]:
import pandas as pd

panel = pd.read_csv(f"{DATA_DIR}/sequences/kinase_panel.csv")
print(panel.shape)
panel.head()

In [ ]:
panel["kinase_length"] = panel["kinase_end"] - panel["kinase_start"] + 1
panel.groupby("organism")["length"].agg(["count", "mean"]).round(0)

In [ ]:
human = panel[panel["organism"] == "Homo sapiens"].sort_values("length", ascending=False)
human[["gene", "protein_name", "length", "kinase_start", "kinase_end", "kinase_length"]]

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 4))
ax.barh(panel["entry_name"], panel["length"], color="lightgray", label="full protein")
ax.barh(panel["entry_name"], panel["kinase_length"], left=panel["kinase_start"], color="tab:blue",
        label="kinase domain")
ax.set_xlabel("residue number"); ax.invert_yaxis(); ax.legend(loc="lower right")
ax.set_title("Where the kinase domain sits in each protein")
plt.tight_layout(); plt.show()

## 4. Biopython: installed and imported

On Colab the setup cell above installed Biopython. Biopython is organized in sub-packages:
`Bio.Seq`, `Bio.SeqIO`, `Bio.Entrez`, `Bio.Align`, `Bio.Blast`, `Bio.PDB`, `Bio.Phylo`, and more.
Always record the version you used; APIs evolve.

In [ ]:
import Bio
from Bio import SeqIO

print("Biopython version:", Bio.__version__)

`SeqIO.parse` reads a sequence file record by record. Here we read the full-length
kinase sequences (FASTA format) and compare lengths with the table above.

In [ ]:
for record in SeqIO.parse(f"{DATA_DIR}/sequences/kinase_panel_full.fasta", "fasta"):
    print(f"{record.id:<22} {len(record.seq):>5} aa   {record.seq[:25]}...")

### ✏️ Exercise 1

Write a function `at_percent(seq)` that returns the percentage of A and T bases, and check
that `gc_percent(dna) + at_percent(dna)` equals 100.

In [ ]:
# Your code here

<details>
<summary><b>Show solution</b></summary>

```python
def at_percent(seq):
    return 100 * (seq.count("A") + seq.count("T")) / len(seq)

print(at_percent(dna), gc_percent(dna) + at_percent(dna))
```

</details>

### ✏️ Exercise 2

Using `panel`, list the genes whose kinase domain starts before residue 500.

In [ ]:
# Your code here

<details>
<summary><b>Show solution</b></summary>

```python
panel.loc[panel["kinase_start"] < 500, ["gene", "organism", "kinase_start"]]
```

</details>

### ✏️ Exercise 3

Count how many records in `kinase_panel_full.fasta` are longer than 1,200 amino acids,
using `SeqIO.parse` and a loop (not pandas).

In [ ]:
# Your code here

<details>
<summary><b>Show solution</b></summary>

```python
n_long = 0
for rec in SeqIO.parse(f"{DATA_DIR}/sequences/kinase_panel_full.fasta", "fasta"):
    if len(rec.seq) > 1200:
        n_long += 1
print(n_long, "proteins are longer than 1,200 aa")
```

</details>

## Key takeaways

- Python indices start at 0; biological numbering starts at 1.
- Dictionaries count, functions reuse, pandas summarizes tables.
- Biopython is imported as `Bio`; record `Bio.__version__` in your projects.

**Next:** [02 · Seq objects](02_seq_objects.ipynb).